In [39]:
from references import zeng24_config, zeng24_question
from src.retrieval import VectorRetriever, RerankerManager, VectorRetriever_fromcfg
from src.prompts import LLMQueryRewriter
from src.utils import get_llm_output_file
import json
import os
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 初始化设置以及数据库

In [2]:
cfg = zeng24_config.Zeng24fiqa()

In [3]:
# 初始化
retriever = VectorRetriever_fromcfg(cfg, device='cpu', force_rebuild=False)

[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/vector-chroma
[INFO] Loading existing Chroma DB: ./data/fiqa


/workspace/zms/Data/rag-llm/src/retrieval.py:73: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(embedding_function=embed_model,


Retriever of mmr is ready.
Retriever of vector-chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!


# 生成或加载问题

In [4]:
# 输入查询
queries = ["Tell me about APPLE.", "Tell me about Google."]

In [7]:
qrw = LLMQueryRewriter(model="./Models/Qwen2.5-14B-Instruct", base_url="http://localhost:22999/v1", api_key="EMPTY")

In [8]:
queries_rws = qrw.rewrite(queries, n_variants=5)

In [9]:
queries_rws

{'original_query': ['Tell me about APPLE.', 'Tell me about Google.'],
 'rewritten_queries': [['What can you tell me about Apple Inc?',
   'What is the history of Apple?',
   "What are some criticisms of Apple's business practices?",
   'How does Apple compare to other technology companies in terms of innovation?',
   'What are the environmental impacts associated with Apple products?'],
  ['What is the history of Google?',
   'How did Google start and grow into what it is today?',
   "What are some criticisms of Google's business practices?",
   'How does Google compare to other search engines in terms of user privacy?',
   'What alternatives exist for search engines that do not use Google?']],
 'all_queries': [['Tell me about APPLE.',
   'What can you tell me about Apple Inc?',
   'What is the history of Apple?',
   "What are some criticisms of Apple's business practices?",
   'How does Apple compare to other technology companies in terms of innovation?',
   'What are the environmenta

# 检索得到chunk

In [14]:
reranker = RerankerManager(reranker_dir=cfg.retrieval.rerank, top_n=10, device='cuda:1')

[INFO] Reranker BAAI/bge-reranker-large is ready!


In [15]:
# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries_rws["original_query"])

# 查看结果
for i, q in enumerate(queries_rws["original_query"]):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: Tell me about APPLE.
  1. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  2. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  3. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  4. [515651] The benefits Apple offers are pretty amazing. 15% discount on stock (you can allocate up to 10% of y...
  5. [488869] Apple is eating the lunch of Nokia, Rimm, and Sprint. A quick check of their balance sheets and fina...
  6. [115991] "What does your comment have to do with my comment? You say ""Apple only designs stuff"" as if that ...
  7. [470984] Seems pretty nice. My only real complaint with my current MacBook Pro is its heft. I opted for the s...
  8. [28862] "I know this I irrelevant, but whale. Man, what an og username.   I agree with you on the offshore b...

🔍 Query: Tell me about Go

In [16]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



🔍 Query: Tell me about APPLE.
  1. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  2. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  3. [28862] "I know this I irrelevant, but whale. Man, what an og username.   I agree with you on the offshore b...
  4. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  5. [515651] The benefits Apple offers are pretty amazing. 15% discount on stock (you can allocate up to 10% of y...
  6. [115991] "What does your comment have to do with my comment? You say ""Apple only designs stuff"" as if that ...
  7. [488869] Apple is eating the lunch of Nokia, Rimm, and Sprint. A quick check of their balance sheets and fina...
  8. [470984] Seems pretty nice. My only real complaint with my current MacBook Pro is its heft. I opted for the s...

🔍 Query: Tell me about Go

#### 下面测试使用rewriter的格式

In [17]:
# 调用 retrieve 方法
contexts, doc_ids = retriever.retrieve(queries_rws["all_queries"])

# 查看结果
for i, q in enumerate(queries_rws["all_queries"]):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: ['Tell me about APPLE.', 'What can you tell me about Apple Inc?', 'What is the history of Apple?', "What are some criticisms of Apple's business practices?", 'How does Apple compare to other technology companies in terms of innovation?', 'What are the environmental impacts associated with Apple products?']
  1. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  2. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  3. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  4. [515651] The benefits Apple offers are pretty amazing. 15% discount on stock (you can allocate up to 10% of y...
  5. [488869] Apple is eating the lunch of Nokia, Rimm, and Sprint. A quick check of their balance sheets and fina...
  6. [115991] "What does your comment have to do with my comment? You say ""Apple only desig

In [18]:
contexts, doc_ids  = reranker.rerank(contexts, doc_ids, queries)

# 查看结果
for i, q in enumerate(queries):
    print(f"\n🔍 Query: {q}")
    for j, (ctx, doc_id) in enumerate(zip(contexts[i], doc_ids[i])):
        print(f"  {j+1}. [{doc_id}] {ctx[:100]}...")


🔍 Query: Tell me about APPLE.
  1. [358413] Apple's specialty is UX.  It's an incredibly talented UX company, both software and hardware wise.  ...
  2. [137360] I can't decide what to do about apple. Huge market share, huge gobs of cash that they don't know wha...
  3. [229950] "The story with most companies, Tesla, Apple, Microsoft, old GM, etc is that innovation was done by ...
  4. [343855] I wouldn't even say it's amazing at UX. Once upon a time it was, but these days it has lost a ton of...
  5. [297512] I mean, they're actually pretty successful in their niche.  And Apple creates a huge positive extern...
  6. [28862] "I know this I irrelevant, but whale. Man, what an og username.   I agree with you on the offshore b...
  7. [172864] For Cook, it was finding Apple, a company that stood for something bigger than selling electronics a...
  8. [162405] I hate apples business practice down to the core, but whenever a not-so-tech-savvy friend or older g...
  9. [515651] The benefits

# 形成结构prompt

In [19]:
from src.llm import OpenAILLM
from src.prompts import SimplePromptConstructor

In [20]:
p_construct = SimplePromptConstructor()

In [21]:
p_construct.prefix

['context: ', 'question: ', 'answer:']

In [22]:
end_ppt = p_construct.batch_construct(queries, contexts)

# 输入LLM进行测试

In [27]:
LL_Model = OpenAILLM(
                    model = "./Models/Qwen2.5-14B-Instruct", 
                    base_url = "http://localhost:22999/v1", 
                    api_key = "EMPTY", 
                    reasoning= cfg.llm.reasoning,
                    temperature= cfg.llm.temperature,
                    top_p= cfg.llm.top_p,
                    max_gen_len= cfg.llm.max_gen_len)

In [28]:
LL_Model.infer("who are you?")

"I'm Qwen, a large language model created by Alibaba Cloud. I'm here to help answer your questions, provide information, and have conversations on a wide range of topics. How can I assist you today?"

In [29]:
LL_Model.batch_infer(end_ppt)

(["Apple Inc., often simply referred to as Apple, is a multinational technology company headquartered in Cupertino, California. Founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne, Apple has grown from a small startup into one of the world's largest and most valuable companies. The company is renowned for its innovative consumer electronics, software, and services, including the iPhone, iPad, Mac computers, Apple Watch, and various software platforms such as macOS, iOS, watchOS, and tvOS.\n\n### Key Points About Apple\n\n1. **Product Line**: Apple's product lineup includes smartphones (iPhone), personal computers (Mac), tablets (iPad), smartwatches (Apple Watch), and streaming devices (Apple TV). Each product is designed to integrate seamlessly with others through Apple's ecosystem, enhancing user experience and convenience.\n\n2. **User Experience (UX)**: Historically, Apple has been celebrated for its focus on user experience, combining sleek design with intuitive interfac

In [30]:
LL_Model.batch_infer(queries)

(['When you mention "APPLE," it\'s likely you\'re referring to Apple Inc., one of the world’s leading technology companies. Founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne, Apple has grown from a small startup to a global giant known for its innovative products and services.\n\n### Key Products and Services\n\n- **iPod**: One of Apple\'s early successes was the iPod, which revolutionized the way people listen to music.\n- **iPhone**: Launched in 2007, the iPhone transformed the smartphone industry with its intuitive touch interface and app ecosystem.\n- **iPad**: Introduced in 2010, the iPad popularized the tablet computer market.\n- **Mac**: Apple\'s line of personal computers, including desktops and laptops, is known for its design and performance.\n- **Apple Watch**: A smartwatch that integrates seamlessly with other Apple devices, offering health tracking and communication features.\n- **Services**: Apple also offers a range of services such as the App Store, Apple M

In [32]:
answers, reasons = LL_Model.batch_infer(end_ppt)

In [41]:
# 保存结果
output_dir = cfg.expconfig.output_dir
os.makedirs(output_dir, exist_ok=True)

answers_path = os.path.join(output_dir, get_llm_output_file(cfg))
with open(answers_path, "w", encoding="utf-8") as f_a:
    json.dump(answers, f_a, ensure_ascii=False, indent=2)

reasons_path = answers_path.replace(".json", "_reasoning.json")
with open(reasons_path, "w", encoding="utf-8") as f_r:
    json.dump(reasons, f_r, ensure_ascii=False, indent=2)

In [42]:
reasons_path

'./exp/fiqa/vector-chroma/bge-large-en-v1_5-Qwen2_5-14B-Instruct/mmr-15-BAAI/bge-reranker-large/outputs-Qwen2.5-14B-Instruct-0-4096-4096_reasoning.json'